# Interpolación Espacial con Kriging
### *Ejemplo Relámpago — Introducción al Análisis de Datos y Programación para el Manejo y Conservación de Recursos Naturales*

---

Este cuaderno cierra la trilogía geoestadística del curso:

| Cuaderno | Herramienta | Pregunta |
|----------|-------------|---------|
| 1 | Correlograma de Moran's I | ¿Hay autocorrelación espacial? ¿A qué escala? |
| 2 | Semivariograma | ¿Cómo se estructura esa autocorrelación? |
| **3** | **Kriging** | **¿Cómo predigo valores en lugares no muestreados?** |

El kriging es el método de interpolación óptimo cuando existe autocorrelación espacial:  
usa el semivariograma ajustado como modelo de la estructura espacial para asignar  
pesos a los puntos vecinos de forma óptima — minimizando la varianza del error de predicción.

> **Intuición clave:** El kriging no solo predice el valor en cada punto no muestreado —  
> también entrega un **mapa de incertidumbre** (varianza de kriging). Saber *dónde no sabemos*  
> es tan valioso como saber dónde sí.

## 0 · Instalación de dependencias

In [ ]:
!pip install numpy scipy matplotlib pykrige scikit-learn --quiet

## 1 · Importaciones

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.spatial.distance import cdist
from scipy.optimize import curve_fit
from pykrige.ok import OrdinaryKriging
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

## 2 · Generar el paisaje de referencia ("la verdad")

En un estudio real nunca conocemos la superficie completa.  
Aquí la generamos nosotros para poder **evaluar la calidad de la interpolación** al final.

Usamos el mismo procedimiento que en los cuadernos anteriores:  
ruido blanco + filtro gaussiano con σ conocido.

In [ ]:
np.random.seed(42)
N_GRID = 100       # grilla N × N
SIGMA  = 12        # escala característica verdadera (celdas)

white_noise = np.random.randn(N_GRID, N_GRID)
landscape   = gaussian_filter(white_noise, sigma=SIGMA)

# Escalar a un rango más interpretable (p.ej. altura de dosel en metros)
landscape = (landscape - landscape.min()) / (landscape.max() - landscape.min()) * 20

# Coordenadas de la grilla (para graficar y comparar)
gx = np.arange(N_GRID)
gy = np.arange(N_GRID)
GX, GY = np.meshgrid(gx, gy)

print(f"Paisaje de referencia: {landscape.shape}, rango [{landscape.min():.2f}, {landscape.max():.2f}] m")

## 3 · Diseño muestral: ¿cuántos puntos y dónde?

Comparamos dos estrategias de muestreo para ver cómo afectan la interpolación:

| Estrategia | Descripción | Ventaja |
|------------|-------------|---------|
| **Aleatorio simple** | Puntos uniformemente al azar | Fácil de implementar |
| **Sistemático** | Grilla regular de puntos | Cobertura homogénea del área |

En ambos casos usamos el mismo número total de puntos.

In [ ]:
N_MUESTRAS = 80   # puntos de muestreo — prueba 30, 80, 200

# ── Muestreo aleatorio ────────────────────────────────────────────────────────
idx_x_rand = np.random.randint(0, N_GRID, N_MUESTRAS)
idx_y_rand = np.random.randint(0, N_GRID, N_MUESTRAS)
z_rand     = landscape[idx_y_rand, idx_x_rand]

# ── Muestreo sistemático (grilla regular) ─────────────────────────────────────
paso  = int(np.sqrt(N_GRID**2 / N_MUESTRAS))
gxs   = np.arange(paso//2, N_GRID, paso)
gys   = np.arange(paso//2, N_GRID, paso)
GXS, GYS = np.meshgrid(gxs, gys)
idx_x_sist = GXS.ravel()
idx_y_sist = GYS.ravel()
z_sist     = landscape[idx_y_sist, idx_x_sist]

print(f"Muestreo aleatorio:     {len(z_rand)} puntos")
print(f"Muestreo sistemático:   {len(z_sist)} puntos")

# Visualizar ambos diseños
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (ix, iy, z, titulo) in zip(axes, [
        (idx_x_rand, idx_y_rand, z_rand, f"Aleatorio (n={len(z_rand)})"),
        (idx_x_sist, idx_y_sist, z_sist, f"Sistemático (n={len(z_sist)})")]):
    ax.imshow(landscape, cmap='terrain', origin='lower', alpha=0.45,
              vmin=0, vmax=20)
    sc = ax.scatter(ix, iy, c=z, cmap='terrain', s=25,
                    edgecolors='k', linewidths=0.4, vmin=0, vmax=20)
    ax.set_title(titulo, fontsize=13)
    ax.set_xlabel('X (celdas)'); ax.set_ylabel('Y (celdas)')
    plt.colorbar(sc, ax=ax, shrink=0.8, label='Altura dosel (m)')
plt.suptitle("Diseños muestrales sobre el paisaje de referencia", fontsize=13)
plt.tight_layout()
plt.savefig('01_disenos_muestrales.png', dpi=150, bbox_inches='tight')
plt.show()

## 4 · ¿Cómo funciona el Kriging Ordinario?

### Predictor lineal óptimo

El kriging predice el valor en un punto no muestreado $\mathbf{x}_0$ como  
una **combinación lineal ponderada** de los $n$ valores observados:

$$\hat{Z}(\mathbf{x}_0) = \sum_{i=1}^{n} \lambda_i \, Z(\mathbf{x}_i)$$

Los pesos $\lambda_i$ se calculan resolviendo el **sistema de kriging**:

$$\begin{pmatrix} \gamma_{11} & \cdots & \gamma_{1n} & 1 \\ \vdots & \ddots & \vdots & \vdots \\ \gamma_{n1} & \cdots & \gamma_{nn} & 1 \\ 1 & \cdots & 1 & 0 \end{pmatrix} \begin{pmatrix} \lambda_1 \\ \vdots \\ \lambda_n \\ \mu \end{pmatrix} = \begin{pmatrix} \gamma_{10} \\ \vdots \\ \gamma_{n0} \\ 1 \end{pmatrix}$$

donde $\gamma_{ij} = \gamma(\|\mathbf{x}_i - \mathbf{x}_j\|)$ es la semivarianza entre los puntos muestreados  
y $\gamma_{i0} = \gamma(\|\mathbf{x}_i - \mathbf{x}_0\|)$ es la semivarianza entre cada muestra y el punto a predecir.

### Varianza de kriging (incertidumbre)

El sistema también entrega la **varianza del error de predicción**:

$$\sigma^2_K(\mathbf{x}_0) = \sum_{i=1}^{n} \lambda_i \, \gamma_{i0} + \mu$$

- Baja cerca de los puntos muestreados → alta confianza  
- Alta lejos de cualquier muestra → baja confianza

> Esta varianza es el argumento más poderoso del kriging frente a otros interpoladores  
> (IDW, splines): no solo predice, sino que **cuantifica la incertidumbre espacial**.

## 5 · Ajustar el semivariograma

Antes de interpolar necesitamos el modelo de semivariograma — el mismo paso del cuaderno anterior,  
aquí aplicado a los datos de muestreo aleatorio.

In [ ]:
# Usaremos el muestreo aleatorio para el kriging principal
coords_rand = np.column_stack([idx_x_rand, idx_y_rand]).astype(float)

# Semivariograma empírico por bandas anulares
dists_m = cdist(coords_rand, coords_rand)
lag_centers  = np.arange(3, 55, 3)
gamma_emp    = []

for d in lag_centers:
    mask = np.triu((dists_m <= d) & (dists_m > d - 3), k=1)
    ii, jj = np.where(mask)
    if len(ii) < 5:
        gamma_emp.append(np.nan); continue
    gamma_emp.append(np.sum((z_rand[ii] - z_rand[jj])**2) / (2 * len(ii)))

gamma_emp = np.array(gamma_emp)

# Modelos teóricos
def modelo_gaussiano(h, nugget, sill, rango):
    return np.where(h == 0, 0,
           nugget + (sill - nugget) * (1 - np.exp(-(h / rango)**2)))

def modelo_esferico(h, nugget, sill, rango):
    return np.where(h <= rango,
           nugget + (sill - nugget) * (1.5*(h/rango) - 0.5*(h/rango)**3),
           np.where(h == 0, 0, sill))

def modelo_exponencial(h, nugget, sill, rango):
    return np.where(h == 0, 0,
           nugget + (sill - nugget) * (1 - np.exp(-h / rango)))

mv   = ~np.isnan(gamma_emp)
sill_est = np.var(z_rand)
resultados_sv = {}
for nombre, modelo in [("gaussian",     modelo_gaussiano),
                        ("spherical",    modelo_esferico),
                        ("exponential",  modelo_exponencial)]:
    try:
        popt, _ = curve_fit(modelo, lag_centers[mv], gamma_emp[mv],
                            p0=[0, sill_est, 20],
                            bounds=([0, 0, 1], [sill_est, sill_est*2, 80]),
                            maxfev=8000)
        res  = gamma_emp[mv] - modelo(lag_centers[mv], *popt)
        rmse = np.sqrt(np.mean(res**2))
        resultados_sv[nombre] = {"popt": popt, "rmse": rmse}
    except Exception as e:
        print(f"  {nombre}: ajuste fallido ({e})")

mejor_sv = min(resultados_sv, key=lambda k: resultados_sv[k]["rmse"])
nugget_m, sill_m, rango_m = resultados_sv[mejor_sv]["popt"]

print(f"Mejor modelo de semivariograma: {mejor_sv}")
print(f"  Pepita  (nugget): {nugget_m:.4f}")
print(f"  Meseta  (sill):   {sill_m:.4f}")
print(f"  Rango   (range):  {rango_m:.1f} celdas")

# Graficar
h_smooth = np.linspace(0.1, 60, 300)
modelos_plot = {"gaussian": modelo_gaussiano, "spherical": modelo_esferico,
                "exponential": modelo_exponencial}
colores_sv   = {"gaussian": "#2E7D32", "spherical": "#E65100", "exponential": "#1565C0"}

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(lag_centers[mv], gamma_emp[mv], color='#37474F', s=40, zorder=5,
           label='Empírico')
for nombre, res in resultados_sv.items():
    lw = 2.5 if nombre == mejor_sv else 1.5
    ls = '-'  if nombre == mejor_sv else '--'
    ax.plot(h_smooth, modelos_plot[nombre](h_smooth, *res["popt"]),
            color=colores_sv[nombre], linewidth=lw, linestyle=ls,
            label=f"{nombre.capitalize()} (RMSE={res['rmse']:.4f}{'  ←' if nombre==mejor_sv else ''})")
ax.axvline(rango_m, color='#D32F2F', linestyle=':', linewidth=1.8,
           label=f'Rango ≈ {rango_m:.1f} celdas')
ax.set_xlabel('Distancia de rezago (celdas)', fontsize=12)
ax.set_ylabel('Semivarianza γ(h)', fontsize=12)
ax.set_title('Semivariograma empírico + modelos ajustados', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('02_semivariograma_kriging.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 · Interpolar con Kriging Ordinario

Usamos `pykrige` — la implementación estándar en Python — con el modelo de semivariograma ajustado.  
Le pasamos directamente los parámetros que obtuvimos en el paso anterior.

In [ ]:
# Grilla de predicción (toda la ventana, cada celda)
grid_x = np.arange(0, N_GRID, 1, dtype=float)
grid_y = np.arange(0, N_GRID, 1, dtype=float)

# Construir el kriging con el mejor modelo y parámetros ajustados
OK = OrdinaryKriging(
    x            = idx_x_rand.astype(float),
    y            = idx_y_rand.astype(float),
    z            = z_rand,
    variogram_model   = mejor_sv,
    variogram_parameters = {
        "nugget": nugget_m,
        "sill":   sill_m,
        "range":  rango_m
    },
    verbose       = False,
    enable_plotting = False
)

# Predecir en toda la grilla
z_pred, sigma2_pred = OK.execute("grid", grid_x, grid_y)
sigma_pred = np.sqrt(sigma2_pred)   # desviación estándar (en unidades originales)

print(f"Superficie interpolada: {z_pred.shape}")
print(f"Rango de predicciones:  [{z_pred.min():.2f}, {z_pred.max():.2f}] m")
print(f"Incertidumbre media:    {sigma_pred.mean():.3f} m (σ kriging)")

## 7 · Comparar: paisaje verdadero vs. interpolado vs. incertidumbre

El panel de tres columnas es la forma canónica de presentar resultados de kriging:  
1. **Verdad** — lo que queremos estimar (en la práctica, desconocido)  
2. **Predicción** — la superficie interpolada por kriging  
3. **Incertidumbre** — la desviación estándar de kriging en cada celda

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
vmin, vmax = 0, 20

# Panel 1: paisaje verdadero
im0 = axes[0].imshow(landscape, cmap='terrain', origin='lower',
                     vmin=vmin, vmax=vmax)
axes[0].scatter(idx_x_rand, idx_y_rand, c='white', s=12,
                edgecolors='k', linewidths=0.5, zorder=5, label='Muestras')
axes[0].set_title('Paisaje verdadero', fontsize=13)
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')
plt.colorbar(im0, ax=axes[0], shrink=0.8, label='Altura dosel (m)')
axes[0].legend(fontsize=9)

# Panel 2: predicción kriging
im1 = axes[1].imshow(z_pred, cmap='terrain', origin='lower',
                     vmin=vmin, vmax=vmax)
axes[1].scatter(idx_x_rand, idx_y_rand, c='white', s=12,
                edgecolors='k', linewidths=0.5, zorder=5)
axes[1].set_title(f'Kriging Ordinario ({mejor_sv.capitalize()})', fontsize=13)
axes[1].set_xlabel('X')
plt.colorbar(im1, ax=axes[1], shrink=0.8, label='Altura dosel (m)')

# Panel 3: incertidumbre
im2 = axes[2].imshow(sigma_pred, cmap='YlOrRd', origin='lower')
axes[2].scatter(idx_x_rand, idx_y_rand, c='white', s=12,
                edgecolors='k', linewidths=0.5, zorder=5)
axes[2].set_title('Incertidumbre (σ kriging)', fontsize=13)
axes[2].set_xlabel('X')
plt.colorbar(im2, ax=axes[2], shrink=0.8, label='σ predicción (m)')

plt.suptitle(f"Kriging Ordinario — n = {N_MUESTRAS} muestras, σ_verdadero = {SIGMA} celdas",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('03_kriging_resultado.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Mapa de error de predicción

El error real es $\hat{Z}(\mathbf{x}) - Z(\mathbf{x})$ — la diferencia entre la predicción  
y el valor verdadero en cada celda. En la práctica esto no se puede calcular (no conoces la verdad),  
pero aquí lo hacemos para evaluar la calidad del kriging.

In [ ]:
error = z_pred - landscape

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mapa de error
lim = np.abs(error).max()
im  = axes[0].imshow(error, cmap='RdBu_r', origin='lower',
                     vmin=-lim, vmax=lim)
axes[0].scatter(idx_x_rand, idx_y_rand, c='black', s=10,
                marker='x', linewidths=0.8, zorder=5, label='Muestras')
axes[0].set_title('Error de predicción (kriging − verdad)', fontsize=13)
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')
plt.colorbar(im, ax=axes[0], shrink=0.8, label='Error (m)')
axes[0].legend(fontsize=9)

# Histograma de errores
axes[1].hist(error.ravel(), bins=50, color='#455A64', edgecolor='white',
             linewidth=0.4)
axes[1].axvline(0, color='#C62828', linewidth=1.5, linestyle='--',
                label='Error = 0')
axes[1].axvline(error.mean(), color='#F57C00', linewidth=1.5, linestyle='-',
                label=f'Media = {error.mean():.3f} m')
rmse_total = np.sqrt(np.mean(error**2))
axes[1].set_title(f'Distribución del error  (RMSE = {rmse_total:.3f} m)', fontsize=13)
axes[1].set_xlabel('Error (m)'); axes[1].set_ylabel('Frecuencia')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('04_error_kriging.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"RMSE global:          {rmse_total:.4f} m")
print(f"Sesgo medio:          {error.mean():.4f} m")
print(f"Error máximo abs.:    {np.abs(error).max():.4f} m")

## 9 · Validación cruzada Leave-One-Out (LOO-CV)

En la práctica **no conocemos el paisaje verdadero** — no podemos calcular el error real.  
La estrategia estándar es la validación cruzada:

1. Retirar un punto de muestra $i$
2. Predecir $\hat{Z}(\mathbf{x}_i)$ con los $n-1$ puntos restantes
3. Calcular el residuo $z_i - \hat{Z}(\mathbf{x}_i)$
4. Repetir para todos los puntos
5. Evaluar RMSE y sesgo del conjunto de residuos

Un kriging bien ajustado produce residuos LOO **sin sesgo** (media ≈ 0) y con  
varianza similar a la varianza de kriging predicha.

In [ ]:
print("Ejecutando validación cruzada LOO...")
n = len(z_rand)
residuos_loo  = np.zeros(n)
sigma_loo     = np.zeros(n)

for i in range(n):
    # Retirar punto i
    idx_train = [j for j in range(n) if j != i]
    x_tr = idx_x_rand[idx_train].astype(float)
    y_tr = idx_y_rand[idx_train].astype(float)
    z_tr = z_rand[idx_train]

    ok_loo = OrdinaryKriging(
        x_tr, y_tr, z_tr,
        variogram_model=mejor_sv,
        variogram_parameters={"nugget": nugget_m, "sill": sill_m, "range": rango_m},
        verbose=False, enable_plotting=False
    )
    z_hat, var_hat = ok_loo.execute(
        "points",
        np.array([float(idx_x_rand[i])]),
        np.array([float(idx_y_rand[i])])
    )
    residuos_loo[i] = z_rand[i] - z_hat[0]
    sigma_loo[i]    = np.sqrt(var_hat[0])
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{n} completado")

rmse_loo = np.sqrt(np.mean(residuos_loo**2))
sesgo_loo = residuos_loo.mean()
print(f"\n✓ LOO-CV completado")
print(f"  RMSE LOO:  {rmse_loo:.4f} m")
print(f"  Sesgo LOO: {sesgo_loo:.4f} m")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residuos LOO vs valor observado
axes[0].scatter(z_rand, z_rand - residuos_loo, alpha=0.6,
                color='#1565C0', edgecolors='k', linewidths=0.3, s=40)
lim_plot = [z_rand.min() - 0.5, z_rand.max() + 0.5]
axes[0].plot(lim_plot, lim_plot, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Valor observado (m)', fontsize=12)
axes[0].set_ylabel('Valor predicho LOO (m)', fontsize=12)
axes[0].set_title(f'Observado vs. Predicho  (RMSE={rmse_loo:.3f} m)', fontsize=13)
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.2)
axes[0].set_xlim(lim_plot); axes[0].set_ylim(lim_plot)
axes[0].set_aspect('equal')

# Residuos estandarizados
res_std = residuos_loo / sigma_loo
axes[1].hist(res_std, bins=20, color='#455A64', edgecolor='white', linewidth=0.4,
             density=True)
# Curva normal de referencia
from scipy.stats import norm
xx = np.linspace(-4, 4, 200)
axes[1].plot(xx, norm.pdf(xx), 'r-', linewidth=2, label='N(0,1) esperada')
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.6)
axes[1].set_xlabel('Residuo estandarizado', fontsize=12)
axes[1].set_ylabel('Densidad', fontsize=12)
axes[1].set_title('Residuos LOO estandarizados', fontsize=13)
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.2)

plt.suptitle("Validación cruzada Leave-One-Out", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('05_validacion_cruzada.png', dpi=150, bbox_inches='tight')
plt.show()

## 10 · Efecto del tamaño muestral sobre la incertidumbre

¿Cuántas muestras necesito para obtener una interpolación aceptable?  
Calculamos el RMSE y la incertidumbre media del kriging para distintos $n$.

In [ ]:
n_lista = [20, 40, 80, 150, 300]
rmse_n  = []
sigma_n = []

print("Evaluando distintos tamaños muestrales...")
for n_eval in n_lista:
    np.random.seed(123)
    ix = np.random.randint(0, N_GRID, n_eval)
    iy = np.random.randint(0, N_GRID, n_eval)
    zv = landscape[iy, ix]

    try:
        ok_n = OrdinaryKriging(
            ix.astype(float), iy.astype(float), zv,
            variogram_model=mejor_sv,
            variogram_parameters={"nugget": nugget_m, "sill": sill_m, "range": rango_m},
            verbose=False, enable_plotting=False
        )
        zp, sp = ok_n.execute("grid", grid_x, grid_y)
        rmse_n.append(np.sqrt(np.mean((zp - landscape)**2)))
        sigma_n.append(np.sqrt(sp).mean())
        print(f"  n={n_eval:>4}: RMSE={rmse_n[-1]:.3f} m, σ_kriging={sigma_n[-1]:.3f} m")
    except Exception as e:
        rmse_n.append(np.nan); sigma_n.append(np.nan)
        print(f"  n={n_eval}: error — {e}")

fig, ax1 = plt.subplots(figsize=(9, 5))
color1, color2 = '#C62828', '#1565C0'
ax1.plot(n_lista, rmse_n, 'o-', color=color1, linewidth=2.5, markersize=8,
         label='RMSE real')
ax1.set_xlabel('Número de muestras (n)', fontsize=13)
ax1.set_ylabel('RMSE (m)', fontsize=13, color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax2 = ax1.twinx()
ax2.plot(n_lista, sigma_n, 's--', color=color2, linewidth=2.5, markersize=8,
         label='σ kriging medio')
ax2.set_ylabel('σ kriging medio (m)', fontsize=13, color=color2)
ax2.tick_params(axis='y', labelcolor=color2)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11)
ax1.set_title('Calidad del kriging en función del tamaño muestral', fontsize=13)
ax1.grid(True, alpha=0.2)
ax1.axvline(N_MUESTRAS, color='gray', linestyle=':', alpha=0.7,
            label=f'n = {N_MUESTRAS} (este análisis)')
plt.tight_layout()
plt.savefig('06_efecto_n_muestras.png', dpi=150, bbox_inches='tight')
plt.show()

## 11 · Kriging vs. IDW — ¿por qué importa el semivariograma?

La interpolación por **distancia inversa ponderada (IDW)** es el método más simple:  
cada punto no muestreado recibe un promedio ponderado de sus vecinos,  
con pesos proporcionales a $1/d^p$.

Comparamos ambos para ver la diferencia en práctica.

In [ ]:
def idw(ix, iy, z_obs, grid_x, grid_y, power=2):
    """Interpolación por distancia inversa ponderada (IDW)."""
    GX2, GY2 = np.meshgrid(grid_x, grid_y)
    pts_obs  = np.column_stack([ix, iy])
    pts_pred = np.column_stack([GX2.ravel(), GY2.ravel()])
    dists    = cdist(pts_pred, pts_obs)
    np.fill_diagonal(dists, 1e-10)  # evitar división por cero
    pesos    = 1.0 / dists**power
    pesos   /= pesos.sum(axis=1, keepdims=True)
    z_idw    = (pesos * z_obs).sum(axis=1)
    return z_idw.reshape(GX2.shape)

z_idw = idw(idx_x_rand, idx_y_rand, z_rand, grid_x, grid_y)
rmse_idw = np.sqrt(np.mean((z_idw - landscape)**2))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
vmin, vmax = 0, 20
titulos = [
    ("Verdad",                              landscape, None),
    (f"Kriging (RMSE={rmse_total:.3f} m)",  z_pred,    None),
    (f"IDW     (RMSE={rmse_idw:.3f} m)",    z_idw,     None),
]
for ax, (titulo, superficie, _) in zip(axes, titulos):
    im = ax.imshow(superficie, cmap='terrain', origin='lower', vmin=vmin, vmax=vmax)
    ax.scatter(idx_x_rand, idx_y_rand, c='white', s=10,
               edgecolors='k', linewidths=0.4, zorder=5)
    ax.set_title(titulo, fontsize=12)
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    plt.colorbar(im, ax=ax, shrink=0.8, label='m')

plt.suptitle(f"Kriging vs. IDW  (n = {N_MUESTRAS} muestras)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('07_kriging_vs_idw.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"RMSE Kriging: {rmse_total:.4f} m")
print(f"RMSE IDW:     {rmse_idw:.4f} m")
diff = (rmse_idw - rmse_total) / rmse_idw * 100
print(f"Mejora del kriging sobre IDW: {diff:.1f}%")

## 12 · Resumen: el kriging en contexto

### Ventajas del kriging frente a otros interpoladores

| Propiedad | IDW | Splines | **Kriging** |
|-----------|-----|---------|-------------|
| Usa la estructura espacial real | ✗ | ✗ | **✓** |
| Interpolador exacto (pasa por las muestras) | ✓ | ✓ | **✓** |
| Mapa de incertidumbre | ✗ | ✗ | **✓** |
| Óptimo en el sentido de mínima varianza | ✗ | ✗ | **✓** |
| Requiere ajustar un semivariograma | ✗ | ✗ | Sí |

### Supuestos del kriging ordinario

1. **Estacionariedad de segundo orden** — la media y la varianza son constantes en el área  
2. **Isotropía** — la estructura espacial es igual en todas las direcciones  
   (si no, se usa kriging anisotrópico)  
3. **El semivariograma es conocido** — en la práctica lo estimamos, lo que introduce incertidumbre adicional

### Relación con el resto del curso

```
Moran's I ──→ ¿Hay autocorrelación? ¿A qué escala?
      │
      ▼
Semivariograma ──→ ¿Cuál es el modelo de la estructura espacial?
      │              (pepita, meseta, rango)
      ▼
Kriging ──→ Predicción óptima + mapa de incertidumbre
```

El semivariograma no es un fin en sí mismo — es el **puente** entre la descripción  
de la autocorrelación y la predicción espacial.

## 13 · Extra: experimenta con los parámetros 🔬

Cambia `N_MUESTRAS` y `SIGMA` en la celda 2 y vuelve a ejecutar el cuaderno completo  
para explorar cómo afectan la calidad del kriging y el mapa de incertidumbre.

También puedes cambiar `power` en la función IDW (§11) para ver cómo reacciona ese método.

In [ ]:
# ── Experimenta aquí ──────────────────────────────────────────────────────────
# Cambia estos valores y ejecuta solo esta celda (requiere haber corrido §6 antes)
N_MUESTRAS_EXP = 40    # prueba 20, 60, 200
SIGMA_EXP      = 6     # prueba 4, 15, 25
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(7)
land_e = (lambda l: (l - l.min())/(l.max() - l.min()) * 20)(
          gaussian_filter(np.random.randn(N_GRID, N_GRID), sigma=SIGMA_EXP))

ix_e = np.random.randint(0, N_GRID, N_MUESTRAS_EXP)
iy_e = np.random.randint(0, N_GRID, N_MUESTRAS_EXP)
z_e  = land_e[iy_e, ix_e]

ok_e = OrdinaryKriging(ix_e.astype(float), iy_e.astype(float), z_e,
                       variogram_model=mejor_sv,
                       variogram_parameters={"nugget": nugget_m, "sill": sill_m, "range": rango_m},
                       verbose=False, enable_plotting=False)
zp_e, sp_e = ok_e.execute("grid", grid_x, grid_y)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (titulo, sup) in zip(axes, [
        ("Verdad",          land_e),
        ("Kriging",         zp_e),
        ("Incertidumbre σ", np.sqrt(sp_e))]):
    cmap = 'terrain' if titulo != "Incertidumbre σ" else 'YlOrRd'
    im   = ax.imshow(sup, cmap=cmap, origin='lower')
    ax.scatter(ix_e, iy_e, c='white', s=12, edgecolors='k', linewidths=0.4, zorder=5)
    ax.set_title(titulo, fontsize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)
rmse_e = np.sqrt(np.mean((zp_e - land_e)**2))
plt.suptitle(f"Experimento — n={N_MUESTRAS_EXP}, σ={SIGMA_EXP}, RMSE={rmse_e:.3f} m",
             fontsize=13, y=1.02)
plt.tight_layout(); plt.show()